# DreamSnap — Free Flux LoRA Training

Trains a Flux LoRA from a model's uploaded images **for free** on a Colab/Kaggle GPU, then uploads the weights back to your app and marks the model `COMPLETED`.

## Before you start
1. In the DreamSnap app, fill the form + upload images and click **Train**. You'll get a **Model ID** (also visible in the Models list).
2. Runtime → **Change runtime type → GPU** (T4 is the free option).
3. Get a Hugging Face token with access to `black-forest-labs/FLUX.1-dev`:
   - Accept the license at https://huggingface.co/black-forest-labs/FLUX.1-dev
   - Create a token at https://huggingface.co/settings/tokens

> **Reality check:** Flux is a 12B model. On a free T4 (16 GB) training is slow (often 1–3 h for ~1000 steps) and can OOM. **Kaggle** (free P100/2×T4, 30 GPU-hrs/week) is usually more reliable. Lower `STEPS` or `resolution` if you hit out-of-memory.

## 1. Configure — edit these

In [ ]:
BACKEND_URL  = "https://dreamsnap.onrender.com"   # your deployed backend
MODEL_ID     = "PASTE_MODEL_ID_HERE"              # from the app after clicking Train
HF_TOKEN     = "hf_xxxxxxxxxxxxxxxx"               # token with FLUX.1-dev access
TRIGGER_WORD = ""        # optional; leave blank to use the model name from the backend
STEPS        = 1000      # 800-1500 typical; lower = faster/cheaper on VRAM
RESOLUTION   = [512, 768]  # drop to [512] if you hit out-of-memory

## 2. Install ai-toolkit

In [ ]:
!nvidia-smi -L
%cd /content
![ -d ai-toolkit ] || git clone https://github.com/ostris/ai-toolkit.git
%cd /content/ai-toolkit
!git submodule update --init --recursive
!pip install -q -r requirements.txt
!pip install -q requests pyyaml huggingface_hub

## 3. Fetch this model's images from the backend

In [ ]:
import os, requests

r = requests.get(f"{BACKEND_URL}/ai/training/{MODEL_ID}/data", timeout=60)
r.raise_for_status()
data = r.json()
print("model:", data["name"], "| status:", data["status"], "| images:", len(data["imageUrls"]))

trigger = (TRIGGER_WORD or data.get("triggerWord") or "subject").strip()
DATASET_DIR = "/content/dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

for i, url in enumerate(data["imageUrls"]):
    img = requests.get(url, timeout=120); img.raise_for_status()
    ext = url.split("?")[0].split(".")[-1].lower()
    if ext not in ("jpg", "jpeg", "png", "webp"):
        ext = "jpg"
    open(os.path.join(DATASET_DIR, f"img_{i}.{ext}"), "wb").write(img.content)
    # one caption file per image = just the trigger word
    open(os.path.join(DATASET_DIR, f"img_{i}.txt"), "w").write(trigger)

print(f"downloaded {len(data['imageUrls'])} images | trigger word: '{trigger}'")

## 4. Log in to Hugging Face (Flux dev is gated)

In [ ]:
from huggingface_hub import login
login(token=HF_TOKEN)

## 5. Write the training config

In [ ]:
import yaml

NAME = f"lora_{MODEL_ID[:8]}"
os.makedirs("/content/output", exist_ok=True)

config = {
    "job": "extension",
    "config": {
        "name": NAME,
        "process": [{
            "type": "sd_trainer",
            "training_folder": "/content/output",
            "device": "cuda:0",
            "trigger_word": trigger,
            "network": {"type": "lora", "linear": 16, "linear_alpha": 16},
            "save": {"dtype": "float16", "save_every": STEPS, "max_step_saves_to_keep": 1},
            "datasets": [{
                "folder_path": DATASET_DIR,
                "caption_ext": "txt",
                "caption_dropout_rate": 0.05,
                "shuffle_tokens": False,
                "cache_latents_to_disk": True,
                "resolution": RESOLUTION,
            }],
            "train": {
                "batch_size": 1,
                "steps": STEPS,
                "gradient_accumulation_steps": 1,
                "train_unet": True,
                "train_text_encoder": False,
                "gradient_checkpointing": True,
                "noise_scheduler": "flowmatch",
                "optimizer": "adamw8bit",
                "lr": 1e-4,
                "dtype": "bf16",
            },
            "model": {
                "name_or_path": "black-forest-labs/FLUX.1-dev",
                "is_flux": True,
                "quantize": True,    # 8-bit base so it fits on a T4
                "low_vram": True,
            },
            "sample": {
                "sampler": "flowmatch",
                "sample_every": STEPS,
                "width": 512, "height": 512,
                "prompts": [f"{trigger} portrait, high quality"],
                "sample_steps": 20,
            },
        }],
    },
    "meta": {"name": NAME, "version": "1.0"},
}

with open("/content/config.yaml", "w") as f:
    yaml.dump(config, f, sort_keys=False)
print(open("/content/config.yaml").read())

## 6. Train (the long step)

In [ ]:
%cd /content/ai-toolkit
!python run.py /content/config.yaml

## 7. Upload the LoRA back to DreamSnap and mark the model COMPLETED

In [ ]:
import glob

safetensors = sorted(glob.glob(f"/content/output/{NAME}/*.safetensors"))
assert safetensors, "No .safetensors produced — check the training logs above."
lora_path = safetensors[-1]
size_mb = os.path.getsize(lora_path) / 1e6
print(f"uploading {lora_path} ({size_mb:.1f} MB)")

# 1) ask the backend for a presigned S3 upload URL
g = requests.post(f"{BACKEND_URL}/api/get-upload-url",
                  json={"fileName": f"{NAME}.safetensors", "fileType": "application/octet-stream"},
                  timeout=60)
g.raise_for_status()
up = g.json()

# 2) PUT the weights to S3
with open(lora_path, "rb") as f:
    put = requests.put(up["uploadURL"], data=f,
                       headers={"Content-Type": "application/octet-stream"}, timeout=1800)
put.raise_for_status()

# 3) tell the backend the model is trained
c = requests.post(f"{BACKEND_URL}/ai/training/complete",
                  json={"modelId": MODEL_ID, "loraUrl": up["publicURL"]}, timeout=60)
c.raise_for_status()
print("\u2705 done:", c.json())
print("LoRA URL:", up["publicURL"])